# Iron Find Electric v1 Kaggle Staging

This notebook is a minimal staging entry point for the repaired benchmark package. It validates the frozen artifact bundle, shows the current benchmark card, and loads the frozen manifests used by the local source-of-truth implementation.

In [ ]:
from pathlib import Path
import sys

CANDIDATE_ROOTS = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    candidate
    for candidate in CANDIDATE_ROOTS
    if (candidate / "src").is_dir()
    and (candidate / "packaging/kaggle/frozen_artifacts_manifest.json").is_file()
)

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from kaggle import load_kaggle_staging_manifest, resolve_kaggle_artifact_path, validate_kaggle_staging_manifest
from splits import PARTITIONS, load_frozen_split, load_split_manifest

validate_kaggle_staging_manifest(REPO_ROOT)
manifest = load_kaggle_staging_manifest(REPO_ROOT)
manifest


In [ ]:
import json

card_path = resolve_kaggle_artifact_path(
    manifest["entry_points"]["benchmark_card"]["path"],
    repo_root=REPO_ROOT,
)
r13_path = resolve_kaggle_artifact_path(
    manifest["evidence_reports"]["anti_shortcut_validity_gate_r13"]["path"],
    repo_root=REPO_ROOT,
)
r15_path = resolve_kaggle_artifact_path(
    manifest["evidence_reports"]["empirical_reaudit_r15"]["path"],
    repo_root=REPO_ROOT,
)

card_excerpt = "\n".join(card_path.read_text(encoding="utf-8").splitlines()[:20])
r13_report = json.loads(r13_path.read_text(encoding="utf-8"))
r15_report = json.loads(r15_path.read_text(encoding="utf-8"))

{
    "benchmark_card": str(card_path.relative_to(REPO_ROOT)),
    "benchmark_card_excerpt": card_excerpt,
    "r13_gate_passed": r13_report["passed"],
    "r13_summary": r13_report["comparison_summary"],
    "r15_audit_note": r15_report["audit_note"],
    "current_emitted_difficulty_labels": manifest["current_emitted_difficulty_labels"],
    "reserved_difficulty_labels": manifest["reserved_difficulty_labels"],
}


In [ ]:
split_summary = {
    partition: {
        "manifest_version": load_split_manifest(partition).manifest_version,
        "seed_bank_version": load_split_manifest(partition).seed_bank_version,
        "episode_split": load_split_manifest(partition).episode_split.value,
        "episode_count": len(load_frozen_split(partition)),
    }
    for partition in PARTITIONS
}
split_summary


## Staging Notes

- Binary is the leaderboard-primary task.
- Narrative remains the robustness companion.
- Post-shift Probe Accuracy is the primary metric.
- The notebook stages the frozen repaired implementation; it does not supersede local validation.